# MeChess 15/15: reference rates for the coach (CPU notebook)

An adapter around `chessme coach-reference-games`, `chessme analyse` and `chessme coach-reference-build`. The coach's weakness profile can tell you where you give away points more often than your own average; this notebook produces what it needs to also say *than players of your rating*: counts of moves, mistakes (at least 0.10 expected points lost) and blunders (at least 0.20) from public Lichess games, by speed (blitz, rapid), 200-point rating band, phase, situation and kind of mistake.

1. Pick `PER_BAND` games per speed and band from byte windows of one month of the Lichess database (standard, rated, no bots, players within 200 points of each other, at least 20 half-moves). Only the headers of the games that are not needed are parsed.
2. Analyse them with Stockfish at **exactly the coach's settings** (`NODES`, `MULTIPV`), so a player's reviewed games and the reference are measured the same way.
3. Reduce to counts (no games, no names) in `reference_rates.json`, which ships with the coach (`chessme/coach/`).

**Before you run:** *Accelerator -> None*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**. About 5,000 games at 100,000 nodes and five lines per position: expect a few hours on four cores. Stockfish is an external download (GPL-3.0, official release); nothing is redistributed.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
MONTH = "2026-06"                                       # a month on https://database.lichess.org
PARTS = 700
WINDOWS = "14,43,72,101,130,159,188,217,246,275,304,333,362,391,420,449,478,507,536,565,594,623,652,681"
PER_BAND = 300                                          # games per speed (blitz, rapid) and 200-point rating band
NODES, MULTIPV = 100000, 5                              # EXACTLY the coach's analysis settings (`chessme coach --nodes`, MultiPV 5): rates from other settings do not compare
WORKERS = 4
STOCKFISH_URL = "https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar"
SOURCE = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"
OUT = "/kaggle/working/coach" if os.path.exists("/kaggle") else "coach_local"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

## Step 1. Stockfish (external, pinned release)

In [ ]:
os.makedirs(OUT, exist_ok=True)
os.makedirs("/tmp/sf", exist_ok=True)
sh("curl", "-fsSL", "-o", "/tmp/sf/sf.tar", STOCKFISH_URL)
sh("tar", "-xf", "/tmp/sf/sf.tar", "-C", "/tmp/sf")
ENGINE = next(f for f in glob.glob("/tmp/sf/**/stockfish*", recursive=True) if os.path.isfile(f) and not f.endswith((".tar", ".txt", ".md")) and os.access(f, os.X_OK))
sh("sh", "-c", f"echo quit | {ENGINE} | head -1 | tee {OUT}/stockfish_version.txt")
sh("sha256sum", "/tmp/sf/sf.tar")

## Step 2. Pick the games

In [ ]:
sh(*CLI, "coach-reference-games", "--source", SOURCE, "--parts", str(PARTS), "--windows", WINDOWS, "--per-band", str(PER_BAND), "--out", f"{OUT}/reference_games.pgn")

## Step 3. Analyse them (resumable: one file per game)

In [ ]:
sh(*CLI, "analyse", f"{OUT}/reference_games.pgn", "--out", f"{OUT}/analysis", "--engine", ENGINE, "--nodes", str(NODES), "--multipv", str(MULTIPV),
   "--workers", str(WORKERS), "--openings-dir", "", "--log", f"{OUT}/analyse.log", "--control-dir", "/tmp/control")

## Step 4. Counts

In [ ]:
sh(*CLI, "coach-reference-build", "--pgn", f"{OUT}/reference_games.pgn", "--analysis", f"{OUT}/analysis", "--nodes", str(NODES), "--multipv", str(MULTIPV),
   "--source", f"Lichess standard rated, {MONTH}", "--out", f"{OUT}/reference_rates.json")
shutil.rmtree(f"{OUT}/analysis/games", ignore_errors=True)       # keep the output small: the counts are what ships